# MCM perturbation 데이터 둘러보기

두 GEO 데이터셋의 로컬 파일을 열어 **유전자 행, 표본 열, 발현량 단위와 조건 라벨**을 확인한다. 각 표의 개수는 파일의 **열 수**이며, 실제 독립 표본 수나 pairing은 GEO와 실험 메타데이터로 확정한다. 이 노트북에서는 로컬 원본을 읽어 구조와 품질 지표를 미리 본다.

**읽는 순서** · 파일 목록 → GSE228927 (mouse, FPKM) → GSE282665 (human, TPM) → 데이터 간 비교

[GEO 파일 다운로드와 재실행 안내](README.md)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "perturbation_validation").is_dir()),
    None,
)
if repo is None:
    raise FileNotFoundError("저장소 루트 또는 notebooks/perturbation에서 실행하세요.")
data = repo / "data" / "perturbation_validation"

inventory = pd.DataFrame([
    {
        "dataset": path.parent.name,
        "file": path.name,
        "MiB": round(path.stat().st_size / 2**20, 2),
        "format": "Excel" if path.suffix == ".xlsx" else "tab-delimited text",
    }
    for path in sorted(data.glob("GSE*/*"))
    if path.is_file() and path.suffix.lower() in {".xlsx", ".xls", ".tsv"}
])
display(inventory)

## GSE228927 · Mouse astrocyte

`.xls` 확장자를 쓰지만 파일은 **탭으로 구분된 텍스트**다. 각 행은 유전자이고, 표본 열에는 FPKM이 들어 있다. 뒤쪽에는 설명, GO, pathway 등의 유전자 주석이 있다. **`NOT1–3`는 resting-MCM 처리군**이며 LPS와 hemin 비교에서 같은 대조군을 사용한다.

In [ ]:
table_228 = pd.read_csv(
    data / "GSE228927" / "GSE228927_FPKM_anno.xls", sep="\t", low_memory=False
).set_index("id")
headers_228 = table_228.columns.to_series().str.extract(r"^(?P<label>Hemin|LPS|NOT)(?P<suffix>\d+)$").dropna()
samples_228 = headers_228.rename_axis("expression_column").reset_index()
samples_228["condition"] = samples_228["label"].map({
    "Hemin": "hemin-MCM", "LPS": "LPS-MCM", "NOT": "resting-MCM"
})
fpkm_228 = table_228.loc[:, samples_228["expression_column"]]
annotation_228 = table_228.drop(columns=fpkm_228.columns)

display(samples_228)
display(samples_228.groupby("condition", sort=False).size().rename("file_columns").to_frame())
display(fpkm_228.head().round(3))
print("Annotation columns:", ", ".join(annotation_228.columns))

## GSE282665 · Human astrocyte와 motor neuron

두 파일 모두 `gene_id`(Ensembl ID)와 `gene_name`(symbol) 뒤에 Salmon gene-level **TPM** 표본 열이 있다. 열 이름에서 stimulus와 배지 라벨의 위치가 서로 다르다.

| Recipient | 원본 패턴 | 원본 열 예 |
| --- | --- | --- |
| Astrocyte | `prefix_media_stimulus` | `ctrl1_ctrl_lps`, `ctrl1_nil_untx` |
| Motor neuron | `prefix_stimulus_media` | `ctrl3_lps_ctrl`, `ctrl3_untx_fresh` |

아래 표의 `ctrl`, `vcp`, `fresh`, `nil`은 **파일 헤더에 있는 라벨**이다. `ctrl` 배지 후보가 motor neuron에서 조건당 6열로 보이지만, 논문 Fig. S7의 healthy-control 비교는 조건당 3 line을 보고한다. 따라서 접두어와 배지 라벨은 후보 열을 찾는 데 사용하며, recipient/source genotype과 pairing은 메타데이터로 확인한다. 별도의 microglia siRNA 파일은 파일 목록과 크기 비교에 포함한다. [Clarke et al., Fig. S7](https://doi.org/10.1186/s13024-024-00773-1)

In [ ]:
gse282 = data / "GSE282665"
human_tables = {
    "astrocyte": pd.read_csv(
        gse282 / "GSE282665_astrocyte.GSE282665_microglia.media.salmon.merged.gene_tpm.tsv",
        sep="\t", low_memory=False, keep_default_na=False,
    ).set_index("gene_id"),
    "motor neuron": pd.read_csv(
        gse282 / "GSE282665_motor.neuron.GSE282665_microglia.media.salmon.merged.gene_tpm.tsv",
        sep="\t", low_memory=False, keep_default_na=False,
    ).set_index("gene_id"),
    "microglia siRNA": pd.read_csv(
        gse282 / "GSE282665_microglia.cxcr4.gpnmb.sirna.salmon.merged.gene_tpm.tsv",
        sep="\t", low_memory=False, keep_default_na=False,
    ).set_index("gene_id"),
}

human_expression = {
    recipient: table.drop(columns="gene_name") for recipient, table in human_tables.items()
}
samples_282 = {}
for recipient, columns in (
    ("astrocyte", ["prefix", "media_label", "stimulus"]),
    ("motor neuron", ["prefix", "stimulus", "media_label"]),
):
    names = human_expression[recipient].columns.to_series(index=None)
    labels = names.str.rsplit("_", n=2, expand=True)
    labels.columns = columns
    labels.insert(0, "expression_column", names)
    labels["condition"] = labels["media_label"] + " / " + labels["stimulus"]
    samples_282[recipient] = labels.reset_index(drop=True)

    print(f"{recipient} · 헤더 라벨별 열 수")
    display(pd.crosstab(labels["media_label"], labels["stimulus"]))
    # Pivot the exact column names so both conditions can be inspected side by side.
    display(labels.pivot(index=["prefix", "media_label"], columns="stimulus", values="expression_column"))
    first_cols = human_expression[recipient].columns[:3]
    display(human_tables[recipient].loc[:, ["gene_name", *first_cols]].head().round(3))

print("microglia siRNA · 원본 열 순서")
print(", ".join(human_expression["microglia siRNA"].columns))

## 파일 구조와 표본별 발현 확인

`gene_rows`는 원본에서 읽은 유전자 행 수다. 인간 TPM 파일에서 동일한 symbol이 여러 Ensembl ID에 붙는 경우를 살펴보기 위해 ID와 symbol의 중복 수를 따로 표시한다. 아래 그래프는 선형 발현 행렬에서 값이 **0보다 큰 유전자 비율**을 표본 열마다 보여준다. 데이터셋 간 유전자 목록과 정량화 기준이 다르므로 점은 각 패널 안에서 비교한다.

In [ ]:
matrix_specs = [
    ("GSE228927 / astrocyte", fpkm_228, "FPKM", None),
    *[(f"GSE282665 / {recipient}", expression, "TPM", human_tables[recipient]["gene_name"])
      for recipient, expression in human_expression.items()],
]
matrix_summary = []
for label, expression, scale, symbols in matrix_specs:
    values = expression.to_numpy(dtype=float)
    invalid = ~np.isfinite(values)
    matrix_summary.append({
        "matrix": label,
        "scale": scale,
        "gene_rows": len(expression),
        "expression_columns": expression.shape[1],
        "repeated_gene_key_rows": expression.index.duplicated().sum(),
        "repeated_symbol_rows": (pd.NA if symbols is None
                                 else symbols[symbols.ne("")].duplicated().sum()),
        "nonfinite_values": invalid.sum(),
        "negative_values": (values < 0).sum(),
        "min": np.nanmin(values),
        "max": np.nanmax(values),
    })
display(pd.DataFrame(matrix_summary).round(3))

In [ ]:
def plot_detected_fraction(ax, expression, labels, title):
    labels = labels.reindex(expression.columns)
    if labels.isna().any():
        raise ValueError(f"{title}: condition labels do not match expression columns")
    order = labels.drop_duplicates().tolist()
    fraction = expression.gt(0).mean(axis=0) * 100
    for pos, condition in enumerate(order):
        values = fraction[labels.eq(condition)]
        offsets = np.linspace(-0.13, 0.13, len(values)) if len(values) > 1 else [0]
        ax.scatter(pos + offsets, values, s=27, alpha=0.9)
    ax.set_xticks(range(len(order)), order, rotation=30, ha="right")
    ax.set_ylabel("Genes with expression > 0 (%)")
    ax.set_title(title, loc="left")
    ax.grid(axis="y", color="0.9")
    ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
plot_detected_fraction(
    axes[0], fpkm_228, samples_228.set_index("expression_column")["condition"],
    "GSE228927 / astrocyte (FPKM)",
)
for ax, recipient in zip(axes[1:], ("astrocyte", "motor neuron")):
    plot_detected_fraction(
        ax, human_expression[recipient], samples_282[recipient].set_index("expression_column")["condition"],
        f"GSE282665 / {recipient} (TPM)",
    )
plt.show()

다음 분석에서는 여기서 확인한 열을 GEO sample ID 및 생물학적 실험 단위에 연결한다. 이후 ACHES, SNAP-a, SNAP-n 점수에 사용할 최종 조건과 표본을 `sample_manifest.tsv`에 기록한다.